<a href="https://colab.research.google.com/github/bandi137/w12-trendfigyelo/blob/main/W12_trendfigyelo_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 – Kutatási trendfigyelő RAG-gal

**Mit csinál a notebook:** letölti az arXiv egy kategóriájának absztraktjait havi bontásban, vektorokká alakítja őket,
kérdésekre forrásmegjelöléssel válaszol (RAG), és kimutatja a felfutó témákat.

A kód a repó `trendfigyelo/` mappájában van; ez a notebook csak beállít, meghív és megjelenít.
**Kódfrissítés után futtasd újra az első cellát, majd az alatta lévőket** (*Runtime → Run after*) – az első cella lehúzza és újratölti a legfrissebb kódot.

**Indítás előtt (egyszer):**
1. *Runtime → Change runtime type → T4 GPU*
2. Bal oldalt a 🔑 *Secrets* fülön legyen egy `GEMINI_API_KEY` nevű titok (kulcs: https://aistudio.google.com/apikey), notebook-hozzáféréssel.
3. *Runtime → Run all*. Az első futás (letöltés + vektorizálás) kb. 15–25 perc, utána a Drive-ról töltődik minden.
   Új hónap lezárulása után csak az az egy hónap töltődik le (`FRISSITES` kapcsoló az 1. szakaszban).

## 0. Kód frissítése a GitHubról

In [ ]:
REPO_URL = "https://github.com/bandi137/w12-trendfigyelo"
!if [ -d /content/repo ]; then git -C /content/repo fetch -q origin && git -C /content/repo reset -q --hard origin/main; else git clone -q {REPO_URL} /content/repo; fi
!git -C /content/repo log -1 --format="Kódverzió: %h – %s (%cr)"
!pip install -q -r /content/repo/requirements.txt
import sys
if "/content/repo" not in sys.path:
    sys.path.insert(0, "/content/repo")
# A korábban betöltött modulok törlése, hogy a git pull után a friss kód töltődjön be
# (az IPython autoreload a Colab Pythonjával nem működik: "No module named 'imp'")
for name in [m for m in sys.modules if m == "trendfigyelo" or m.startswith("trendfigyelo.")]:
    del sys.modules[name]
import trendfigyelo as tf

## 1. Beállítások

In [ ]:
CATEGORY   = "cs.CL"        # arXiv kategória
MONTHS     = 12             # hány hónapra visszamenőleg
PER_MONTH  = 400            # cikk / hónap (12 x 400 = 4800 absztrakt)
FRISSITES  = "auto"         # "auto": csak a hiányzó hónapot tölti le | "mindig": mindent újra | "soha": a meglévőt használja
EMB_MODEL  = "intfloat/multilingual-e5-base"   # többnyelvű -> magyar kérdés is működik
LLM_MODEL  = "gemini-3.5-flash-lite"           # ha 404-et ad, nézd meg az aktuális nevet az AI Studióban
# Technikai tesztekhez gemini-3.5-flash-lite (gyors, olcsó); a RAG minőségi teszteléséhez gemini-3.1-pro-preview (erősebb érvelés, fizetős API).
TOP_K      = 5              # ennyi cikket kap meg a nyelvi modell
N_TOPICS   = 25             # témák száma a trendelemzéshez
N_SHOW     = 8              # ennyi téma kerül a táblázatba

import os
from google.colab import drive, userdata
from google import genai
drive.mount("/content/drive")
DATA_DIR = f"/content/drive/MyDrive/W12_trendfigyelo/{CATEGORY.replace('.', '_')}"
os.makedirs(DATA_DIR, exist_ok=True)
llm = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Adatmappa:", DATA_DIR)

## 2. Adatgyűjtés az arXivról (hónaponként a Drive-on tárolva)

Egy lezárt hónap cikkei nem változnak, ezért minden hónap egyszer töltődik le (`honapok/ÉÉÉÉ-HH.csv`, a letöltés dátuma a `letoltes.json`-ban).
Amikor lezárul egy új hónap, `FRISSITES = "auto"` mellett csak az töltődik le, a legrégebbi kiesik az időablakból.

In [ ]:
df = tf.load_or_download(DATA_DIR, CATEGORY, MONTHS, PER_MONTH, refresh=FRISSITES)
df.head()

## 3. Vektorizálás

Ha a cikklista és a modell ugyanaz, mint legutóbb, a vektorok a Drive-ról töltődnek. Ha bármelyik változott (új hónap, más `EMB_MODEL`),
az összes vektor újraszámolódik (T4 GPU-n néhány perc).

In [ ]:
embedder = tf.Embedder(EMB_MODEL)
emb = tf.load_or_embed(df, embedder, DATA_DIR)
emb.shape

## 4. Kérdés–válasz forrásmegjelöléssel (RAG)

In [ ]:
rag = tf.RAG(df, emb, embedder.query, llm, LLM_MODEL, top_k=TOP_K)
rag.ask("Milyen módszereket javasolnak mostanában a nagy nyelvi modellek hallucinációinak csökkentésére?");

In [ ]:
rag.ask("What are recent approaches to evaluating LLM agents?");

## 5. Trendek: témák, nevek, havi részarány

In [ ]:
df["topic"], keywords = tf.find_topics(emb, (df.title + ". " + df.abstract).values, n_topics=N_TOPICS)
share, growth = tf.topic_trends(df.month, df.topic)
topic_ids = [int(t) for t in growth.index]          # növekedés szerint csökkenő sorrendben

names = tf.name_topics(llm, LLM_MODEL, df, keywords, topic_ids, CATEGORY, DATA_DIR)
table = tf.trend_table(df, topic_ids, names, keywords, growth)
table.to_csv(f"{DATA_DIR}/trendek.csv", index=False)
display(tf.style_table(table, N_SHOW))

In [ ]:
tf.plot_trends(share, growth, names, topic_ids, CATEGORY, path=f"{DATA_DIR}/trendek.png", n=N_SHOW);

### Tématérkép

Minden pont egy cikk: a vektorokat a UMAP vetíti síkba, így a hasonló cikkek egymás mellé kerülnek. A szín a cikk témájának növekedése (piros: nő, kék: csökken). A csúszkával egy hónap cikkei emelhetők ki, a ▶ gomb végigjátssza a hónapokat.
A koordináták elsőre 1–2 perc alatt számolódnak, és a Drive-ra mentődnek (`terkep.npy`); amíg a vektorok nem változnak, onnan töltődnek.

In [ ]:
coords = tf.map_coords(emb, df.id, DATA_DIR)
tf.topic_map(coords, df, names, growth, CATEGORY).show()

In [ ]:
print(tf.summarize_trends(llm, LLM_MODEL, names, growth, topic_ids, CATEGORY, n=N_SHOW))

## 6. Kiértékelés

A tesztkérdések a repó [`eval/kerdesek.csv`](https://github.com/bandi137/w12-trendfigyelo/blob/main/eval/kerdesek.csv)
fájljában vannak (útmutató: `eval/README.md`). A fájl a GitHub webes felületén szerkeszthető; a notebook első cellája
minden futáskor lehúzza a friss változatot, ezért szerkesztés után az első cellát és ezt a szakaszt kell újrafuttatni.

In [ ]:
# Segéd: várt cikkek keresése egy tesztkérdéshez. Írd át a kérdést és a kulcsszót, majd futtasd.
# Csak azt az azonosítót írd a kerdesek.csv-be, amelyik a cím és az absztrakt alapján tényleg válaszol a kérdésre.
KERDES    = "Milyen tanítás nélküli módszerek csökkentik a képi nyelvi modellek hallucinációit?"
KULCSSZO  = "hallucination"

import pandas as pd
pd.set_option("display.max_colwidth", 120)
print("A keresés (RAG) top 10 találata:")
display(rag.retrieve(KERDES, k=10)[["id", "title", "month", "score"]])
print(f"Kulcsszavas találatok a címekben ('{KULCSSZO}'):")
display(df[df.title.str.contains(KULCSSZO, case=False)][["id", "title", "month"]].head(20))

In [ ]:
questions = tf.load_questions("/content/repo/eval/kerdesek.csv", known_ids=set(df.id))
if questions:
    eval_df, summary = tf.evaluate(questions, lambda q, k: rag.retrieve(q, k=k))
    eval_df.to_csv(f"{DATA_DIR}/eval.csv", index=False)
    display(eval_df); print(summary)
else:
    print("Még nincs tesztkérdés az eval/kerdesek.csv-ben.")

In [ ]:
# Kérdés–válasz napló a dokumentációhoz (a kézi 1-5 pontozást ebbe a táblába lehet beírni)
rag.save_log(f"{DATA_DIR}/qa_naplo.csv")

## 7. Webes felület (Gradio) és közzététel

Az alábbi cella elindít egy webes felületet. A kimenetben megjelenik egy nyilvános link (`….gradio.live`), ami a
Colab-munkamenet végéig működik, és **jelszó védi**: a 🔑 *Secrets* fülön adj meg egy `UI_PASSWORD` titkot
(felhasználónév: `w12`). Jelszó nélkül a felület bárki számára elérhető, aki ismeri a linket. Fülek: **Kérdezz**, **Trendek**, **Kiértékelés**, **Közzététel**.

A **Közzététel** gomb a jelentést statikus weboldalként feltölti a GitHub Pages-re:
https://bandi137.github.io/w12-trendfigyelo/ — ehhez a 🔑 *Secrets* fülön kell egy `GITHUB_TOKEN`
(útmutató: [`docs/KOZZETETEL.md`](https://github.com/bandi137/w12-trendfigyelo/blob/main/docs/KOZZETETEL.md)).

Elég az 1–3. szakaszt lefuttatni előtte (a 4–6. opcionális); a felület a 4. szakasz `rag` objektumát használja,
így az ott feltett kérdések is bekerülnek a jelentésbe.

In [ ]:
if "rag" not in globals():
    rag = tf.RAG(df, emb, embedder.query, llm, LLM_MODEL, top_k=TOP_K)
app = tf.TrendApp(df, emb, embedder, llm, rag, data_dir=DATA_DIR,
                  settings=dict(category=CATEGORY, months=MONTHS, emb_model=EMB_MODEL, llm_model=LLM_MODEL,
                                n_topics=N_TOPICS, n_show=N_SHOW, top_k=TOP_K))
app.launch(share=True)